In [ ]:
# ============================================================
# 1- Database design in phpMyAdmin -> SQL codes
# ============================================================

In [1]:
# ============================================================
# 2- Phone Number Normalization
# ============================================================

import re

_PERSIAN_DIGITS = str.maketrans(
    "۰۱۲۳۴۵۶۷۸۹",
    "0123456789"
)

_ARABIC_DIGITS = str.maketrans(
    "٠١٢٣٤٥٦٧٨٩",
    "0123456789"
)


def normalize_iran_phone(value) -> str:
    """
    Return an Iranian mobile number in local 10-digit form.

    Example:
        9121234567

    Accepted formats:
        9121234567
        09121234567
        +989121234567
        00989121234567

    Persian and Arabic-Indic digits are also accepted.
    """

    if value is None:
        raise ValueError("Phone number is empty")

    # Normalize Unicode digits first.
    s = str(value).strip()
    s = s.translate(_PERSIAN_DIGITS)
    s = s.translate(_ARABIC_DIGITS)

    # Reject obvious Excel numeric artifacts.
    if re.fullmatch(r"\d+\.0+", s):
        raise ValueError(
            f"Phone number appears to have been converted to a numeric "
            f"Excel value: {value!r}"
        )

    # Keep digits and an optional leading '+'.
    s = re.sub(r"[^\d+]", "", s)

    # Convert international Iranian formats to local format.
    if s.startswith("+98"):
        s = s[3:]
    elif s.startswith("0098"):
        s = s[4:]
    elif s.startswith("98") and len(s) == 12:
        s = s[2:]
    elif s.startswith("0"):
        s = s[1:]

    # Final Iranian mobile validation.
    if not re.fullmatch(r"9\d{9}", s):
        raise ValueError(
            f"Invalid Iranian mobile number: {value!r}"
        )

    return s

In [2]:
# ============================================================
# 3- Import Excel Data into the Database
# ============================================================
from pathlib import Path

import pandas as pd
import mysql.connector

EXCEL_FILE = Path(
    "/home/saeed/repositories/contacts/import_eitaa/contacts-1-44.xlsx"
)


def get_db_connection():
    return mysql.connector.connect(
        host="127.0.0.1",
        port=3306,
        user="reza",
        password="123",
        database="eitaa_import",
        charset="utf8mb4",
    )


def import_excel():
    df = pd.read_excel(
        EXCEL_FILE,
        dtype={
            "Full Name": "string",
            "Phone Number": "string",
        },
    )

    expected = {"Full Name", "Phone Number"}

    if set(df.columns) != expected:
        raise ValueError(
            f"Expected columns {expected}, got {set(df.columns)}"
        )

    conn = get_db_connection()

    try:
        cursor = conn.cursor()

        sql = """
            INSERT INTO contacts (
                full_name,
                phone_original,
                phone_normalized,
                status
            )
            VALUES (%s, %s, %s, 'pending')
            ON DUPLICATE KEY UPDATE
                full_name = VALUES(full_name),
                phone_original = VALUES(phone_original)
        """

        processed = 0
        invalid_phones = 0
        empty_names = 0

        for _, row in df.iterrows():

            # ------------------------------------------------
            # Full name
            # ------------------------------------------------
            if pd.isna(row["Full Name"]):
                empty_names += 1
                continue

            full_name = str(row["Full Name"]).strip()

            if not full_name:
                empty_names += 1
                continue

            # ------------------------------------------------
            # Phone number
            # ------------------------------------------------
            if pd.isna(row["Phone Number"]):
                invalid_phones += 1
                print("Invalid phone number: empty Excel cell")
                continue

            phone_original = str(row["Phone Number"]).strip()

            try:
                phone_normalized = normalize_iran_phone(
                    phone_original
                )
            except ValueError as exc:
                invalid_phones += 1
                print(
                    f"Invalid phone number "
                    f"{phone_original!r}: {exc}"
                )
                continue

            # ------------------------------------------------
            # Insert/update database
            # ------------------------------------------------
            cursor.execute(
                sql,
                (
                    full_name,
                    phone_original,
                    phone_normalized,
                ),
            )

            processed += 1

        conn.commit()

        # ----------------------------------------------------
        # Verify database state after import
        # ----------------------------------------------------
        cursor.execute(
            """
            SELECT
                COUNT(*) AS total,
                SUM(status = 'pending') AS pending,
                SUM(status = 'processing') AS processing,
                SUM(status = 'success') AS success,
                SUM(status = 'skipped_existing') AS skipped_existing,
                SUM(status = 'failed') AS failed
            FROM contacts
            """
        )

        stats = cursor.fetchone()

        print()
        print("========== IMPORT RESULT ==========")
        print(f"Excel rows processed : {processed}")
        print(f"Invalid phones       : {invalid_phones}")
        print(f"Empty names          : {empty_names}")
        print()
        print(f"DB total             : {stats[0]}")
        print(f"DB pending           : {stats[1]}")
        print(f"DB processing        : {stats[2]}")
        print(f"DB success           : {stats[3]}")
        print(f"DB skipped_existing  : {stats[4]}")
        print(f"DB failed            : {stats[5]}")
        print("====================================")

    except Exception:
        conn.rollback()
        raise

    finally:
        cursor.close()
        conn.close()


if __name__ == "__main__":
    import_excel()



========== IMPORT RESULT ==========
Excel rows processed : 44
Invalid phones       : 0
Empty names          : 0

DB total             : 44
DB pending           : 44
DB processing        : 0
DB success           : 0
DB skipped_existing  : 0
DB failed            : 0


In [3]:
# ============================================================
# 4- Chrome session persistence
# ============================================================
# Clean, isolated Selenium Chrome session

from pathlib import Path
from tempfile import mkdtemp
from typing import Optional
import shutil

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.webdriver import WebDriver

# ChromeDriver

CHROMEDRIVER = (
        Path.cwd()
        / "drivers"
        / "chromedriver"
        / "chromedriver"
)

# Create a completely new temporary Chrome User Data Directory

TEMP_CHROME_PROFILE = Path(
    mkdtemp(prefix="selenium-chrome-")
)

# Close previous Selenium session, if one exists

previous_driver: Optional[WebDriver] = globals().get("driver")

if previous_driver is not None:
    try:
        previous_driver.quit()
        print("\nPrevious Selenium driver closed.")
    except Exception as e:
        print("\nPrevious Selenium driver could not be closed:")
        print(repr(e))

# Chrome options

options = Options()

options.add_argument(
    f"--user-data-dir={TEMP_CHROME_PROFILE}"
)

options.add_argument("--start-maximized")

#  Explicit manual ChromeDriver
#  Selenium Manager is not used for driver discovery.


service = Service(
    executable_path=str(CHROMEDRIVER)
)

# Create new isolated Chrome session


driver: WebDriver = webdriver.Chrome(
    service=service,
    options=options,
)

driver.set_page_load_timeout(60)


First run:

driver = create_driver()

driver.get("https://web.eitaa.com/")

Log into Eitaa manually.

Close the browser normally.

On subsequent executions Selenium will reuse that Chrome profile/session.

This is much more robust than trying to automate OTP authentication.

In [5]:
# ============================================================
# 5- Close Selenium and remove temporary Chrome profile
# ============================================================

if "driver" in globals() and driver is not None:
    try:
        driver.quit()
        print("Selenium Chrome session closed.")
    finally:
        driver = None

if "TEMP_CHROME_PROFILE" in globals():
    shutil.rmtree(
        TEMP_CHROME_PROFILE,
        ignore_errors=True
    )

    print(
        "Temporary Selenium Chrome profile removed."
    )

Selenium Chrome session closed.
Temporary Selenium Chrome profile removed.


In [4]:
# ============================================================
# 6- Selenium Selectors
# ============================================================
# To test that the selector is unique:
# allow pasting
# document.querySelectorAll(".btn-menu-item.tgico-user.rp").length

from selenium.webdriver.common.by import By


class Selectors:
    # 1. Hamburger menu
    HAMBURGER_MENU = (
        By.CSS_SELECTOR,
        ".btn-icon.btn-menu-toggle.rp.sidebar-tools-button.is-visible"
    )

    # 2. Contacts menu item
    CONTACTS_BUTTON = (
        By.XPATH,
        "//div[contains(@class, 'btn-menu-item') and .//span[normalize-space()='Contacts']]"
    )

    # 3. "+" button inside Contacts
    ADD_CONTACT_BUTTON = (
        By.CSS_SELECTOR,
        "button.tgico-add.is-visible"
    )

    # 4. First name field
    FIRST_NAME = (
        By.CSS_SELECTOR,
        'div.input-field:first-child > div.input-field-input[contenteditable="true"]'
    )

    # 5. Phone number field
    PHONE = (
        By.CSS_SELECTOR,
        'div.input-field-phone > '
        'div.input-field-input[contenteditable="true"]'
    )

    # 6. Add/Save button
    SAVE_BUTTON = (
        By.CSS_SELECTOR,
        ".popup-create-contact button.btn-primary.btn-color-primary"
    )

    # 7. Close Add Contact window ( × button )
    CLOSE_CONTACT_WINDOW = (
        By.CSS_SELECTOR,
        ".popup-create-contact button.btn-icon.popup-close"
    )

In [5]:
# ============================================================
# 7- Reliable waits
# ============================================================
#
# Use explicit waits instead of time.sleep().
#
# The purpose of this section is to provide reusable,
# reliable synchronization helpers for the Selenium code
# used in later sections.
#
# These helpers are designed for situations such as:
#   - waiting for Chrome/Eitaa elements to appear
#   - waiting for an element to become visible
#   - waiting for an element to become clickable
#   - waiting for an element/popup to disappear
#   - waiting for a specific Selenium condition
#   - handling temporary/stale DOM elements
#
# ============================================================

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    StaleElementReferenceException,
    NoSuchElementException,
    ElementClickInterceptedException,
)

# ------------------------------------------------------------
# General configuration
# ------------------------------------------------------------

WAIT_SECONDS = 30


# ------------------------------------------------------------
# 1. Create a WebDriverWait object
# ------------------------------------------------------------
#
# Keeping this in one function makes it possible to change
# the global timeout later without changing all wait helpers.

def get_wait(driver, timeout=WAIT_SECONDS):
    """
    Return a WebDriverWait instance.

    Parameters
    ----------
    driver : WebDriver
        Active Selenium WebDriver.

    timeout : int | float
        Maximum number of seconds to wait.

    Returns
    -------
    WebDriverWait
    """
    return WebDriverWait(
        driver,
        timeout,
        poll_frequency=0.2,
        ignored_exceptions=(
            NoSuchElementException,
            StaleElementReferenceException,
        ),
    )


# ------------------------------------------------------------
# 2. Wait until an element exists in the DOM
# ------------------------------------------------------------

def wait_present(driver, locator, timeout=WAIT_SECONDS):
    """
    Wait until an element is present in the DOM.

    The element does not necessarily have to be visible.

    Useful when:
        - the element exists but is not yet displayed
        - another wait will handle visibility/clickability
    """

    return get_wait(driver, timeout).until(
        EC.presence_of_element_located(locator)
    )


# ------------------------------------------------------------
# 3. Wait until an element is visible
# ------------------------------------------------------------

def wait_visible(driver, locator, timeout=WAIT_SECONDS):
    """
    Wait until an element is present and visible.

    Returns
    -------
    WebElement
    """

    return get_wait(driver, timeout).until(
        EC.visibility_of_element_located(locator)
    )


# ------------------------------------------------------------
# 4. Wait until an element is clickable
# ------------------------------------------------------------

def wait_clickable(driver, locator, timeout=WAIT_SECONDS):
    """
    Wait until an element is visible and enabled/clickable.

    This should normally be used before clicking buttons,
    links, menu items, etc.
    """

    return get_wait(driver, timeout).until(
        EC.element_to_be_clickable(locator)
    )


# ------------------------------------------------------------
# 5. Wait until an element disappears
# ------------------------------------------------------------

def wait_invisible(driver, locator, timeout=WAIT_SECONDS):
    """
    Wait until an element is no longer visible.

    This is particularly useful for:
        - closing popups
        - waiting for dialogs to disappear
        - waiting for loading overlays to disappear

    Note:
        Selenium's invisibility_of_element_located() also
        succeeds if the element is removed from the DOM.
    """

    return get_wait(driver, timeout).until(
        EC.invisibility_of_element_located(locator)
    )


# ------------------------------------------------------------
# 6. Wait until an element disappears from the DOM
# ------------------------------------------------------------

def wait_stale(driver, element, timeout=WAIT_SECONDS):
    """
    Wait until a previously obtained WebElement becomes stale.

    Useful when an existing element is replaced/removed after
    an operation such as saving a form.
    """

    return get_wait(driver, timeout).until(
        EC.staleness_of(element)
    )


# ------------------------------------------------------------
# 7. Wait for a locator to contain specific text
# ------------------------------------------------------------

def wait_text(driver, locator, text, timeout=WAIT_SECONDS):
    """
    Wait until an element contains the specified text.
    """

    return get_wait(driver, timeout).until(
        EC.text_to_be_present_in_element(
            locator,
            text,
        )
    )


# ------------------------------------------------------------
# 8. Wait for a specific attribute value
# ------------------------------------------------------------

def wait_attribute(
        driver,
        locator,
        attribute,
        value,
        timeout=WAIT_SECONDS,
):
    """
    Wait until an element has the specified attribute value.

    Example:
        wait_attribute(
            driver,
            Selectors.SOME_ELEMENT,
            "class",
            "active",
        )
    """

    return get_wait(driver, timeout).until(
        EC.attribute_to_include(
            locator,
            value,
        )
    )


# ------------------------------------------------------------
# 9. Wait until an element is selected
# ------------------------------------------------------------

def wait_selected(driver, locator, timeout=WAIT_SECONDS):
    """
    Wait until a selectable element is selected.
    """

    return get_wait(driver, timeout).until(
        EC.element_located_to_be_selected(locator)
    )


# ------------------------------------------------------------
# 10. Wait for a custom condition
# ------------------------------------------------------------

def wait_until(
        driver,
        condition,
        timeout=WAIT_SECONDS,
):
    """
    Wait until a custom Selenium condition returns a truthy
    value.

    The condition receives the driver.

    Example:

        wait_until(
            driver,
            lambda d: some_condition(d)
        )

    This is useful when Selenium's built-in expected
    conditions are not sufficient.
    """

    return get_wait(driver, timeout).until(condition)


# ------------------------------------------------------------
# 11. Safe click helper
# ------------------------------------------------------------

def wait_and_click(driver, locator, timeout=WAIT_SECONDS):
    """
    Wait until an element is clickable and then click it.

    Returns
    -------
    WebElement
        The element that was clicked.
    """

    element = wait_clickable(
        driver,
        locator,
        timeout,
    )

    element.click()

    return element


# ------------------------------------------------------------
# 12. Wait for a popup to close
# ------------------------------------------------------------

def wait_popup_closed(
        driver,
        popup_locator,
        timeout=WAIT_SECONDS,
):
    """
    Wait until a popup/dialog is no longer visible.

    This is a convenience wrapper around
    wait_invisible().

    It is intended for situations such as the Eitaa
    Add Contact popup.
    """

    return wait_invisible(
        driver,
        popup_locator,
        timeout,
    )


# ------------------------------------------------------------
# 13. Wait for an element and then enter text
# ------------------------------------------------------------

def wait_and_type(
        driver,
        locator,
        text,
        timeout=WAIT_SECONDS,
        clear_first=True,
):
    """
    Wait for an input field to become visible and type text.

    Parameters
    ----------
    driver : WebDriver
        Active Selenium driver.

    locator : tuple
        Selenium locator, for example:
            (By.CSS_SELECTOR, "...")

    text : str
        Text to enter.

    clear_first : bool
        If True, clear the input before typing.

    Returns
    -------
    WebElement
        The input element.
    """

    element = wait_visible(
        driver,
        locator,
        timeout,
    )

    if clear_first:
        element.clear()

    element.send_keys(text)

    return element


In [6]:
# ============================================================
# 8- Add Contact to Eitaa
# ============================================================

from datetime import datetime
from selenium.webdriver.common.by import By
from selenium.common.exceptions import (
    TimeoutException,
    StaleElementReferenceException,
)

# ------------------------------------------------------------
# Eitaa result message
# ------------------------------------------------------------

UNREGISTERED_MESSAGE = (
    "The person with this phone number is not registered on Eitaa yet."
)


# ------------------------------------------------------------
# Database status update helper
# ------------------------------------------------------------

def update_contact_status(
        phone_normalized,
        status,
        error_message=None,
        increment_attempt=False,
):
    """
    Update the corresponding contact in the database.

    Parameters
    ----------
    phone_normalized : str
        Normalized Iranian mobile number.

    status : str
        One of:
            pending
            processing
            success
            skipped
            failed

    error_message : str | None
        Error information for failed contacts.

    increment_attempt : bool
        If True, increase attempts by one.
    """

    conn = get_db_connection()

    try:
        cursor = conn.cursor()

        if increment_attempt:
            sql = """
                UPDATE contacts
                SET
                    status = %s,
                    attempts = attempts + 1,
                    error_message = %s,
                    completed_at = NULL
                WHERE phone_normalized = %s
            """

            cursor.execute(
                sql,
                (
                    status,
                    error_message,
                    phone_normalized,
                ),
            )

        else:
            sql = """
                UPDATE contacts
                SET
                    status = %s,
                    error_message = %s
                WHERE phone_normalized = %s
            """

            cursor.execute(
                sql,
                (
                    status,
                    error_message,
                    phone_normalized,
                ),
            )

        conn.commit()

    except Exception:
        conn.rollback()
        raise

    finally:
        cursor.close()
        conn.close()


# ------------------------------------------------------------
# Mark contact as processing
# ------------------------------------------------------------

def mark_contact_processing(phone_normalized):
    """
    Mark a contact as processing and record its start time.

    The attempt counter is increased here because every call to
    add_contact() represents one actual Eitaa attempt.
    """

    conn = get_db_connection()

    try:
        cursor = conn.cursor()

        cursor.execute(
            """
            UPDATE contacts
            SET
                status = 'processing',
                attempts = attempts + 1,
                error_message = NULL,
                started_at = NOW(),
                completed_at = NULL
            WHERE phone_normalized = %s
            """,
            (phone_normalized,),
        )

        if cursor.rowcount != 1:
            raise ValueError(
                f"Contact not found in database: {phone_normalized}"
            )

        conn.commit()

    except Exception:
        conn.rollback()
        raise

    finally:
        cursor.close()
        conn.close()


# ------------------------------------------------------------
# Mark final contact result
# ------------------------------------------------------------

def mark_contact_result(
        phone_normalized,
        status,
        error_message=None,
):
    """
    Store the final result of the current Eitaa attempt.

    Final states handled here:
        success
        skipped
        failed
    """

    conn = get_db_connection()

    try:
        cursor = conn.cursor()

        cursor.execute(
            """
            UPDATE contacts
            SET
                status = %s,
                error_message = %s,
                completed_at = NOW()
            WHERE phone_normalized = %s
            """,
            (
                status,
                error_message,
                phone_normalized,
            ),
        )

        if cursor.rowcount != 1:
            raise ValueError(
                f"Contact not found in database: {phone_normalized}"
            )

        conn.commit()

    except Exception:
        conn.rollback()
        raise

    finally:
        cursor.close()
        conn.close()


# ------------------------------------------------------------
# Detect the Eitaa "not registered" toast
# ------------------------------------------------------------

def is_unregistered_contact(driver):
    """
    Return True if Eitaa displays the exact unregistered-contact
    message.

    The toast does not have its own close button. The Add Contact
    window's close button must therefore be clicked twice.
    """

    try:
        elements = driver.find_elements(
            By.XPATH,
            (
                "//*[contains("
                "normalize-space(text()), "
                f"{UNREGISTERED_MESSAGE!r}"
                ")]"
            ),
        )

        for element in elements:
            try:
                if element.is_displayed():
                    return True
            except StaleElementReferenceException:
                continue

    except Exception:
        pass

    return False


# ------------------------------------------------------------
# Close Add Contact window
# ------------------------------------------------------------

def close_add_contact_window(driver):
    """
    Click the Add Contact window's × button.

    For an unregistered contact this function must be called
    twice:

        1. First click dismisses the toast.
        2. Second click closes the Add Contact window.
    """

    close_button = wait_clickable(
        driver,
        Selectors.CLOSE_CONTACT_WINDOW,
    )

    close_button.click()


# ------------------------------------------------------------
# Add one contact to Eitaa
# ------------------------------------------------------------

def add_contact(driver, full_name, phone_normalized):
    """
    Add exactly one contact to Eitaa.

    Parameters
    ----------
    driver : selenium.webdriver.WebDriver
        Active Selenium WebDriver.

    full_name : str
        Contact's full name.

    phone_normalized : str
        Normalized Iranian mobile number, e.g. 9121234567.

    Returns
    -------
    str
        One of:

            "success"
            "skipped"
            "failed"

    Behavior
    --------
    Registered contact:
        success -> no retry.

    Unregistered contact:
        skipped -> no retry.

    Selenium/Eitaa error:
        failed -> Section 10 decides whether another attempt
        should be made.
    """

    # --------------------------------------------------------
    # Validate basic input
    # --------------------------------------------------------

    if not full_name or not str(full_name).strip():
        raise ValueError("Contact name is empty.")

    if not phone_normalized:
        raise ValueError("Normalized phone number is empty.")

    full_name = str(full_name).strip()
    phone_normalized = str(phone_normalized).strip()

    # --------------------------------------------------------
    # Mark this contact as processing.
    #
    # attempts is incremented here.
    # Section 10 will decide whether a failed contact should
    # be attempted again.
    # --------------------------------------------------------

    mark_contact_processing(phone_normalized)

    try:

        # ====================================================
        # 1. Make sure we are on the Contacts page
        # ====================================================

        contacts_containers = driver.find_elements(
            By.CSS_SELECTOR,
            "#contacts-container",
        )

        if contacts_containers:
            contacts_container = contacts_containers[0]
        else:
            contacts_container = None

        contacts_visible = (
                contacts_container is not None
                and "active"
                in contacts_container.get_attribute("class").split()
        )

        if not contacts_visible:
            wait_and_click(
                driver,
                Selectors.HAMBURGER_MENU,
            )

            wait_and_click(
                driver,
                Selectors.CONTACTS_BUTTON,
            )

        # ====================================================
        # 2. Open Add Contact
        # ====================================================

        add_button = wait_clickable(
            driver,
            Selectors.ADD_CONTACT_BUTTON,
        )

        add_button.click()

        # ====================================================
        # 3. Fill first-name/full-name field
        # ====================================================

        wait_and_type(
            driver,
            Selectors.FIRST_NAME,
            full_name,
            clear_first=True,
        )

        # ====================================================
        # 4. Fill phone-number field
        # ====================================================

        wait_and_type(
            driver,
            Selectors.PHONE,
            phone_normalized,
            clear_first=True,
        )

        # ====================================================
        # 5. Click Save
        # ====================================================

        save_button = wait_clickable(
            driver,
            Selectors.SAVE_BUTTON,
        )

        save_button.click()

        # ====================================================
        # 6. Determine Eitaa's result
        #
        # For an unregistered number Eitaa leaves the
        # Add Contact window open and displays the toast.
        # Therefore we check for the exact toast before
        # waiting for the popup to disappear.
        # ====================================================

        try:
            get_wait(driver, WAIT_SECONDS).until(
                lambda d: (
                        is_unregistered_contact(d)
                        or not d.find_elements(
                    By.CSS_SELECTOR,
                    ".popup-create-contact.active",
                )
                )
            )

        except TimeoutException:
            raise RuntimeError(
                "Timed out while waiting for Eitaa to report "
                "the result of the Add Contact operation."
            )

        # ====================================================
        # 7. Unregistered number
        # ====================================================

        if is_unregistered_contact(driver):
            # ------------------------------------------------
            # First click:
            # dismiss the unregistered-number toast.
            # ------------------------------------------------

            close_add_contact_window(driver)

            # ------------------------------------------------
            # Second click:
            # close the Add Contact window itself.
            # ------------------------------------------------

            close_add_contact_window(driver)

            # ------------------------------------------------
            # Confirm that the Add Contact popup is closed.
            # ------------------------------------------------

            wait_popup_closed(
                driver,
                (
                    By.CSS_SELECTOR,
                    ".popup-create-contact.active",
                ),
            )

            # ------------------------------------------------
            # Store "skipped".
            #
            # This is NOT a failure and must NOT be retried.
            # ------------------------------------------------

            mark_contact_result(
                phone_normalized,
                status="skipped",
                error_message=None,
            )

            print(
                f"[SKIPPED] {full_name} - "
                f"{phone_normalized} "
                f"(not registered on Eitaa)"
            )

            return "skipped"

        # ====================================================
        # 8. Successful contact
        #
        # At this point the unregistered toast did not appear
        # and the Add Contact popup has closed.
        # ====================================================

        wait_popup_closed(
            driver,
            (
                By.CSS_SELECTOR,
                ".popup-create-contact.active",
            ),
        )

        # ----------------------------------------------------
        # Store success.
        # ----------------------------------------------------

        mark_contact_result(
            phone_normalized,
            status="success",
            error_message=None,
        )

        print(
            f"[SUCCESS] {full_name} - {phone_normalized}"
        )

        return "success"

    # ========================================================
    # 9. Any unexpected Selenium/Eitaa error
    # ========================================================

    except Exception as exc:

        error_message = (
            f"{type(exc).__name__}: {exc}"
        )

        # ----------------------------------------------------
        # Best-effort cleanup.
        #
        # If the Add Contact window is still open, try to
        # close it so Section 10 can continue with the next
        # attempt/contact.
        # ----------------------------------------------------

        try:
            popup = driver.find_elements(
                By.CSS_SELECTOR,
                ".popup-create-contact.active",
            )

            if popup:
                close_add_contact_window(driver)

                try:
                    wait_popup_closed(
                        driver,
                        (
                            By.CSS_SELECTOR,
                            ".popup-create-contact.active",
                        ),
                        timeout=5,
                    )
                except Exception:
                    pass

        except Exception:
            pass

        # ----------------------------------------------------
        # Store failed.
        #
        # The attempt was already incremented when the contact
        # was marked "processing".
        #
        # Section 10 is responsible for deciding whether this
        # failed contact should be retried.
        # ----------------------------------------------------

        try:
            mark_contact_result(
                phone_normalized,
                status="failed",
                error_message=error_message,
            )
        except Exception as db_exc:
            raise RuntimeError(
                "The Eitaa operation failed and the database "
                "could not be updated. "
                f"Eitaa error: {error_message}; "
                f"Database error: {db_exc}"
            ) from db_exc

        print(
            f"[FAILED] {full_name} - {phone_normalized}"
        )
        print(
            f"         {error_message}"
        )

        return "failed"


In [7]:
# ============================================================
# 9- Fetch Next Contact from Database
# ============================================================

def get_next_contact():
    """
    Fetch the next contact that should be processed.

    Processing order:
        1. New/pending contacts.
        2. Failed contacts with fewer than 3 attempts.

    Returns
    -------
    dict | None
        A dictionary containing:

            id
            full_name
            phone_original
            phone_normalized
            status
            attempts

        Returns None when there are no more contacts to process.

    Notes
    -----
    Section 9 only fetches a contact.

    It does NOT:
        - open Eitaa
        - call add_contact()
        - retry contacts
        - change the contact status

    Section 10 is responsible for calling add_contact() and
    handling the returned result.
    """

    conn = get_db_connection()

    try:
        cursor = conn.cursor(dictionary=True)

        cursor.execute(
            """
            SELECT
                id,
                full_name,
                phone_original,
                phone_normalized,
                status,
                attempts
            FROM contacts
            WHERE
                status = 'pending'
                OR (
                    status = 'failed'
                    AND attempts < 3
                )
            ORDER BY
                CASE
                    WHEN status = 'pending' THEN 0
                    WHEN status = 'failed' THEN 1
                END,
                id ASC
            LIMIT 1
            """
        )

        contact = cursor.fetchone()

        return contact

    finally:
        cursor.close()
        conn.close()


# ------------------------------------------------------------
# Convenience function for displaying the next contact
# ------------------------------------------------------------

def fetch_next_contact():
    """
    Fetch and display the next contact to be processed.

    This is a small wrapper around get_next_contact() and is
    useful for testing Section 9 manually before Section 10
    is implemented.

    Returns
    -------
    dict | None
        The next contact, or None if no contact remains.
    """

    contact = get_next_contact()

    if contact is None:
        print("No contacts remain to be processed.")
        return None

    print()
    print("========== NEXT CONTACT ==========")
    print(f"ID               : {contact['id']}")
    print(f"Full name        : {contact['full_name']}")
    print(f"Phone original   : {contact['phone_original']}")
    print(f"Phone normalized : {contact['phone_normalized']}")
    print(f"Status           : {contact['status']}")
    print(f"Attempts         : {contact['attempts']}")
    print("==================================")

    return contact


In [11]:
# ============================================================
# 10- Main Processing Loop
# ============================================================

# Maximum number of attempts for a failed contact.
MAX_ATTEMPTS = 3


# ------------------------------------------------------------
# Database helpers for import_runs
# ------------------------------------------------------------

def get_total_contacts():
    """
    Return the total number of contacts in the database.
    """

    conn = get_db_connection()

    try:
        cursor = conn.cursor()

        cursor.execute(
            """
            SELECT COUNT(*)
            FROM contacts
            """
        )

        result = cursor.fetchone()

        return int(result[0])

    finally:
        cursor.close()
        conn.close()


def create_import_run():
    """
    Create a new import_runs record.

    Returns
    -------
    int
        ID of the newly created import run.
    """

    total_contacts = get_total_contacts()

    conn = get_db_connection()

    try:
        cursor = conn.cursor()

        cursor.execute(
            """
            INSERT INTO import_runs (
                started_at,
                total_contacts,
                success_count,
                skipped_count,
                failed_count,
                status
            )
            VALUES (
                NOW(),
                %s,
                0,
                0,
                0,
                'running'
            )
            """,
            (total_contacts,),
        )

        run_id = cursor.lastrowid

        conn.commit()

        return run_id

    except Exception:
        conn.rollback()
        raise

    finally:
        cursor.close()
        conn.close()


def update_import_run_counts(
        run_id,
        success_count,
        skipped_count,
        failed_count,
):
    """
    Update the current import_runs counters.
    """

    conn = get_db_connection()

    try:
        cursor = conn.cursor()

        cursor.execute(
            """
            UPDATE import_runs
            SET
                success_count = %s,
                skipped_count = %s,
                failed_count = %s
            WHERE id = %s
            """,
            (
                success_count,
                skipped_count,
                failed_count,
                run_id,
            ),
        )

        conn.commit()

    except Exception:
        conn.rollback()
        raise

    finally:
        cursor.close()
        conn.close()


def finish_import_run(
        run_id,
        success_count,
        skipped_count,
        failed_count,
        status="completed",
):
    """
    Finish the current import run and store its final counters.
    """

    conn = get_db_connection()

    try:
        cursor = conn.cursor()

        cursor.execute(
            """
            UPDATE import_runs
            SET
                completed_at = NOW(),
                success_count = %s,
                skipped_count = %s,
                failed_count = %s,
                status = %s
            WHERE id = %s
            """,
            (
                success_count,
                skipped_count,
                failed_count,
                status,
                run_id,
            ),
        )

        conn.commit()

    except Exception:
        conn.rollback()
        raise

    finally:
        cursor.close()
        conn.close()


# ------------------------------------------------------------
# Main processing loop
# ------------------------------------------------------------

def run_import():
    """
    Process all contacts that are eligible for processing.

    Processing rules
    ----------------
    success:
        Contact was successfully added.
        No retry.

    skipped:
        Phone number is not registered on Eitaa.
        No retry.

    failed:
        Retry until the contact has reached MAX_ATTEMPTS.

    After the third failed attempt:
        The contact remains 'failed' and the program moves
        to the next contact.

    Returns
    -------
    dict
        Final processing statistics.
    """

    run_id = create_import_run()

    success_count = 0
    skipped_count = 0
    failed_count = 0

    processed_results = 0

    print()
    print("============================================================")
    print("EITAA CONTACT IMPORT STARTED")
    print("============================================================")
    print(f"Import run ID : {run_id}")
    print(f"Total contacts: {get_total_contacts()}")
    print(f"Maximum attempts for failures: {MAX_ATTEMPTS}")
    print("============================================================")
    print()

    try:

        while True:

            # ------------------------------------------------
            # Fetch the next eligible contact.
            #
            # Section 9 decides whether a contact is still
            # eligible based on status and attempts.
            # ------------------------------------------------

            contact = get_next_contact()

            # ------------------------------------------------
            # No contact means the entire processing queue is
            # finished.
            # ------------------------------------------------

            if contact is None:
                break

            contact_id = contact["id"]
            full_name = contact["full_name"]
            phone_normalized = contact["phone_normalized"]
            attempts_before = contact["attempts"]

            print()
            print("------------------------------------------------------------")
            print(
                f"Processing contact ID {contact_id}: "
                f"{full_name}"
            )
            print(
                f"Phone: {phone_normalized}"
            )
            print(
                f"Previous attempts: {attempts_before}"
            )
            print("------------------------------------------------------------")

            # ------------------------------------------------
            # One actual Eitaa attempt.
            #
            # Section 8:
            #   - changes status to processing
            #   - increments attempts
            #   - performs the Eitaa operation
            #   - changes status to success/skipped/failed
            # ------------------------------------------------

            result = add_contact(
                driver,
                full_name,
                phone_normalized,
            )

            processed_results += 1

            # ------------------------------------------------
            # SUCCESS
            # ------------------------------------------------

            if result == "success":

                success_count += 1

                print(
                    f"[SUCCESS] Contact ID {contact_id} "
                    f"completed."
                )

            # ------------------------------------------------
            # SKIPPED
            #
            # This means Eitaa reported that the phone number
            # is not registered.
            #
            # It must NEVER be retried.
            # ------------------------------------------------

            elif result == "skipped":

                skipped_count += 1

                print(
                    f"[SKIPPED] Contact ID {contact_id} "
                    f"is not registered on Eitaa."
                )

            # ------------------------------------------------
            # FAILED
            #
            # Section 9 will return this contact again if:
            #
            #     attempts < MAX_ATTEMPTS
            #
            # Therefore we do not call add_contact() again
            # here. The next loop iteration handles the retry.
            # ------------------------------------------------

            elif result == "failed":

                # The attempt counter was already incremented
                # by Section 8.
                current_attempts = attempts_before + 1

                if current_attempts < MAX_ATTEMPTS:

                    print(
                        f"[FAILED] Contact ID {contact_id} "
                        f"failed on attempt "
                        f"{current_attempts}/{MAX_ATTEMPTS}."
                    )

                    print(
                        "         It will be retried."
                    )

                else:

                    failed_count += 1

                    print(
                        f"[FAILED] Contact ID {contact_id} "
                        f"failed on the final attempt "
                        f"{current_attempts}/{MAX_ATTEMPTS}."
                    )

                    print(
                        "         No more retries."
                    )

            # ------------------------------------------------
            # Unexpected return value from Section 8
            # ------------------------------------------------

            else:

                raise RuntimeError(
                    f"add_contact() returned an unexpected "
                    f"result for contact ID {contact_id}: "
                    f"{result!r}"
                )

            # ------------------------------------------------
            # Update import_runs after every completed attempt.
            # ------------------------------------------------

            update_import_run_counts(
                run_id=run_id,
                success_count=success_count,
                skipped_count=skipped_count,
                failed_count=failed_count,
            )

            print()
            print(
                f"Run totals -> "
                f"success: {success_count}, "
                f"skipped: {skipped_count}, "
                f"failed: {failed_count}"
            )

    except KeyboardInterrupt:

        # ----------------------------------------------------
        # User manually stopped the notebook.
        #
        # Do not mark the import as completed.
        # The run is recorded as failed/interrupted.
        # ----------------------------------------------------

        finish_import_run(
            run_id=run_id,
            success_count=success_count,
            skipped_count=skipped_count,
            failed_count=failed_count,
            status="failed",
        )

        print()
        print("============================================================")
        print("IMPORT INTERRUPTED BY USER")
        print("============================================================")
        print(f"Import run ID : {run_id}")
        print(f"Success       : {success_count}")
        print(f"Skipped       : {skipped_count}")
        print(f"Failed        : {failed_count}")
        print("============================================================")

        raise

    except Exception as exc:

        # ----------------------------------------------------
        # Unexpected application/Selenium/database error.
        # ----------------------------------------------------

        try:
            finish_import_run(
                run_id=run_id,
                success_count=success_count,
                skipped_count=skipped_count,
                failed_count=failed_count,
                status="failed",
            )
        except Exception as db_exc:

            print()
            print(
                "WARNING: The import run failed and the "
                "import_runs record could not be updated."
            )
            print(
                f"Database error: {db_exc}"
            )

        print()
        print("============================================================")
        print("IMPORT FAILED")
        print("============================================================")
        print(f"Import run ID : {run_id}")
        print(f"Success       : {success_count}")
        print(f"Skipped       : {skipped_count}")
        print(f"Failed        : {failed_count}")
        print()
        print(
            f"{type(exc).__name__}: {exc}"
        )
        print("============================================================")

        raise

    # --------------------------------------------------------
    # All eligible contacts have been processed.
    # --------------------------------------------------------

    finish_import_run(
        run_id=run_id,
        success_count=success_count,
        skipped_count=skipped_count,
        failed_count=failed_count,
        status="completed",
    )

    print()
    print("============================================================")
    print("EITAA CONTACT IMPORT COMPLETED")
    print("============================================================")
    print(f"Import run ID : {run_id}")
    print(f"Success       : {success_count}")
    print(f"Skipped       : {skipped_count}")
    print(f"Failed        : {failed_count}")
    print(f"Attempts made : {processed_results}")
    print("============================================================")

    return {
        "run_id": run_id,
        "success_count": success_count,
        "skipped_count": skipped_count,
        "failed_count": failed_count,
        "attempts_made": processed_results,
    }


# ============================================================
# Start the import
# ============================================================

results = run_import()




EITAA CONTACT IMPORT STARTED
Import run ID : 1
Total contacts: 44
Maximum attempts for failures: 3


------------------------------------------------------------
Processing contact ID 1: محمد کاظم آب پیکر - لیست نظام
Phone: 9188626037
Previous attempts: 0
------------------------------------------------------------
[FAILED] محمد کاظم آب پیکر - لیست نظام - 9188626037
         TimeoutException: Message: 

[FAILED] Contact ID 1 failed on attempt 1/3.
         It will be retried.

Run totals -> success: 0, skipped: 0, failed: 0

------------------------------------------------------------
Processing contact ID 2: عزت اله آبائی - لیست نظام
Phone: 9181614708
Previous attempts: 0
------------------------------------------------------------
[FAILED] عزت اله آبائی - لیست نظام - 9181614708
         TimeoutException: Message: 

[FAILED] Contact ID 2 failed on attempt 1/3.
         It will be retried.

Run totals -> success: 0, skipped: 0, failed: 0

----------------------------------------------

KeyboardInterrupt: 